# Case-family review for the 114-case MVP

Screen **all 506 frozen source opinions**, review candidate links, and carry family
IDs into the 114-case metadata and every existing source passage. Run all cells
from MVP or any directory inside the repository. Python standard library only;
keep the `family_review/` support folder beside this notebook.

The source pool, selected 114 IDs, target labels, and original review submissions
are unchanged. Outputs are under `data/family_v1/` and
`data/metadata_v2_family_review/`; earlier metadata is preserved.

**Status:** assistant review (`llm_draft`), not human adjudication. A screened
singleton is not proof of independence. Missing companions and canonical publication
choices remain explicit uncertainties. No final splits or encoder chunks are made.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys
import html
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
spec = importlib.util.spec_from_file_location("mvp_family_audit", MVP / "family_review/family_audit.py")
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
print("Family audit version:", audit.CONFIG["version"])


Family audit version: family_v1


## 1. Reproduce candidate generation without labels

All 127,765 unordered source pairs are checked. Candidate triggers are exact and
normalized text hashes; normalized case/plaintiff names; shared numeric docket
identifiers; identical opinion IDs; and text similarity. Dates and day gaps are
recorded for review, without restricting detection to one period.

Similarity uses log-term-frequency TF-IDF cosine (threshold 0.70), five-token-shingle
Jaccard (0.30), or containment (0.60), with at least 50 shared shingles for the latter
two. Terms present in over 80% of documents are omitted from TF-IDF. These are
candidate-screening settings, not proof of a relationship or an outcome model.
Shared legal language can generate false positives. Missing docket metadata and
changed captions can cause false negatives; no exhaustive external search is claimed.

The scanner also collects amendment/reargument/companion-appeal signal contexts.
Ordinary publication correction notices and amended pleadings/orders do not establish
an amended appellate opinion. Source texts are never cleaned or changed here.


In [2]:
screen = audit.screen(ROOT)
previous = audit.read_json(MVP / "family_review/candidate_screen.json")
audit.require(screen == previous, "Screen differs from reviewed candidates; create a new review/version.")
print("Pairs screened:", screen["pair_count_screened"])
print("Candidate pairs:", len(screen["candidates"]))
print("Exact / normalized full-text duplicate pairs:", screen["exact_duplicate_pairs"], screen["normalized_duplicate_pairs"])
print("Opinions with amendment/related-appeal signals:", len(screen["opinion_notices"]))


Pairs screened: 127765
Candidate pairs: 53
Exact / normalized full-text duplicate pairs: 0 0
Opinions with amendment/related-appeal signals: 10


## 2. Read the saved review decisions

`family_review/review_decisions.json` records each pair's relationship, rationale,
uncertainty, evidence quotes/offsets/hashes, and review basis. The review log explicitly
distinguishes full-opinion review (Coreano and the missing-companion cases), review
of all token-difference blocks for publication variants, and caption/incident-excerpt
review for false positives. It does not claim that every opinion was read in full.
`variant_differences.json` retains the complete token differences for the ten variants.

- **10 publication-variant pairs:** matching opinions with front-matter, formatting,
  citation, or OCR differences. Group together; retain both records. Do not call them
  exact text duplicates or automatically choose one as the authoritative version.
- **Coreano:** two distinct January 20, 2021 companion appeals, with the same index
  number and an explicit “decided herewith” reference. Keep together; neither
  replaces the other.
- **42 false positives:** different parties/incidents or premises, or shared legal
  language. Keep separate; supporting identity evidence is retained.
- **Sodhi:** one source record discusses two appeals. No separate companion record
  was found by name/text search in the frozen corpus. Its missing identity and date
  remain unresolved. Rozier, Hartnett, and Gould have similar warnings outside MVP.

No amended/superseding appellate-opinion pair was identified in this snapshot.
This is not verification of external publication history. All decisions await human review.


In [3]:
decisions = audit.read_json(MVP / "family_review/review_decisions.json")
print("Saved pair decisions:", len(decisions["pair_reviews"]))
print("Saved opinion-notice decisions:", len(decisions["opinion_notice_reviews"]))
print(decisions["amendment_review"]["conclusion"])


Saved pair decisions: 53
Saved opinion-notice decisions: 10
No current-appellate-opinion amendment/supersession relationship identified in the frozen corpus. Amended trial orders and complaints are not amended appellate opinions. Publication correction boilerplate is not proof of a distinct amended record.


## 3. Build family and chunk manifests; update metadata

Connected components of accepted links define versioned family IDs. Every source
case gets a group, including explicit provisional singletons. Every source passage
inherits its case's family. Existing A/B/C annotation packets contain several cases;
packet membership is recorded per case, and an entire multi-case packet must **not**
be used as a model input or split unit. Future encoder chunks must carry `case_id`
and use `attach_families_to_chunks` to inherit the same family ID.

Calibration exposure propagates to the entire family. This adds MVP case
`NY-6004283` (Germain), whose publication counterpart `NY-4308801` is a calibration
case. Thus four selected cases are training-only based on recorded exposure.

Observed dates are compared against 2010–2020 / 2021–2022 / 2023–2024. Any detected
family spanning periods is flagged for primary-comparison exclusion. Missing
companions are held for review; their unknown dates are not treated as resolved.
No final split is assigned, and human-reviewed targets remain pending.


In [4]:
built = audit.build_outputs(ROOT, screen)
summary = built["summary"]
print(json.dumps(summary, indent=2))


{
  "source_records": 506,
  "pairs_screened": 127765,
  "candidate_pairs_reviewed": 53,
  "pair_relationship_counts": {
    "unrelated_screening_false_positive": 42,
    "same_opinion_publication_variant": 10,
    "related_litigation_companion_appeals": 1
  },
  "exact_text_duplicate_pairs": 0,
  "normalized_text_duplicate_pairs": 0,
  "confirmed_appellate_amendment_pairs": 0,
  "all_source_families": 495,
  "mvp_cases": 114,
  "mvp_families": 114,
  "linked_families_touching_mvp": 5,
  "known_cross_period_families": 0,
  "unresolved_external_companions": 4,
  "mvp_unresolved_external_companions": [
    "NY-4655053"
  ],
  "mvp_training_only_cases": [
    "NY-2757329",
    "NY-4546774",
    "NY-4840329",
    "NY-6004283"
  ],
  "mvp_newly_inherited_training_only": [
    "NY-6004283"
  ],
  "source_passages_mapped": 1954,
  "annotation_case_memberships_mapped": 506,
  "actual_splits_assigned": 0,
  "human_adjudication": "pending"
}


## 4. Verify coverage and grouping protections

The builder verifies source hashes, evidence offsets, review coverage, connected
components, complete passage coverage, and unchanged MVP IDs/targets. The checks
below exercise future chunk/split protections with synthetic assignments only;
these examples are not an experimental split manifest.


In [5]:
case_manifest = built["case_manifest"]
lookup = {r["case_id"]: r for r in case_manifest}
audit.require(len(case_manifest) == 506 and len(built["metadata"]) == 114, "Coverage failed")
audit.require(lookup["NY-4849013"]["case_family_id"] == lookup["NY-4849014"]["case_family_id"], "Coreano separated")
audit.require(lookup["NY-6004283"]["family_training_only"], "Germain calibration exposure lost")
audit.require(lookup["NY-4655053"]["external_companion_unresolved"], "Sodhi uncertainty lost")

def expect_rejection(fn, expected_text):
    try:
        fn()
    except ValueError as exc:
        audit.require(expected_text in str(exc), f"Unexpected failure: {exc}")
    else:
        raise AssertionError("Unsafe synthetic assignment was accepted")

expect_rejection(lambda: audit.validate_family_partition([
    {"case_id": "NY-4303491", "split": "train"},
    {"case_id": "NY-6228997", "split": "test"}], case_manifest), "multiple splits")
expect_rejection(lambda: audit.validate_family_partition([
    {"case_id": "NY-6004283", "split": "test"}], case_manifest), "Calibration-family leakage")
expect_rejection(lambda: audit.validate_family_partition([
    {"case_id": "NY-4655053", "split": "train"}], case_manifest), "Resolve external companion")
expect_rejection(lambda: audit.attach_families_to_chunks([
    {"case_id": "NY-4849014", "chunk_id": "synthetic:1", "case_family_id": "wrong"}], case_manifest), "Conflicting family")
expect_rejection(lambda: audit.attach_families_to_chunks([
    {"case_id": "unknown", "chunk_id": "synthetic:2"}], case_manifest), "Unknown chunk case ID")
attached = audit.attach_families_to_chunks([
    {"case_id": "NY-4849013", "chunk_id": "synthetic:a"},
    {"case_id": "NY-4849014", "chunk_id": "synthetic:b"}], case_manifest)
audit.require(attached[0]["case_family_id"] == attached[1]["case_family_id"], "Future chunks separated")
print("Coverage, linked-opinion grouping, calibration inheritance, missing-companion hold, and chunk guards passed.")


Coverage, linked-opinion grouping, calibration inheritance, missing-companion hold, and chunk guards passed.


## 5. Inspect the MVP family metadata

All 114 IDs are retained in 114 distinct observed families. Five of those families
also contain a source record outside the MVP. The table shows grouping metadata
only; the exported updated metadata preserves each reviewer's target fields.


In [6]:
columns = ["case_id", "case_name", "case_family_id", "family_member_case_ids",
           "family_review_status", "calibration_training_only", "primary_boundary_policy_status"]
def cell_value(v):
    return "; ".join(v) if isinstance(v, list) else str(v)
head = "<tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in columns) + "</tr>"
body = "".join("<tr>" + "".join(f"<td>{html.escape(cell_value(r[c]))}</td>" for c in columns) + "</tr>"
               for r in built["metadata"])
TABLE_HTML = ('<div style="overflow:auto;max-height:650px"><table border="1" cellpadding="5" '
              'style="border-collapse:collapse;font-size:12px">'
              '<caption>114-case family metadata — assistant draft</caption>'
              f'<thead>{head}</thead><tbody>{body}</tbody></table></div>')
try:
    from IPython.display import HTML, display
except ImportError:
    print("All 114 rows are saved in data/metadata_v2_family_review/metadata.jsonl.")
else:
    display(HTML(TABLE_HTML))


case_id,case_name,case_family_id,family_member_case_ids,family_review_status,calibration_training_only,primary_boundary_policy_status
NY-10090934,Kerr-Inniss v. Williams,FAM-NY-10090934,NY-10090934,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-10128963,"Kavenaght v. 498 Seventh, LLC",FAM-NY-10128963,NY-10128963,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-10160793,Porter v. Stone,FAM-NY-10160793,NY-10160793,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-10291100,"German v. 333 Rector Garage, LLC",FAM-NY-10291100,NY-10291100,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-10309802,Hart v. 210 W. 77 St. LLC,FAM-NY-10309802,NY-10309802,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-2682947,Duluc v. AC & L Food Corp.,FAM-NY-2682947,NY-2682947,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-2686388,"Benjamin v. Trade Fair Supermarket, Inc.",FAM-NY-2686388,NY-2686388,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-2686408,"Mathis v. D.D. Dylan, LLC",FAM-NY-2686408,NY-2686408,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-2719389,"Simon v. Hamlet Windwatch Development, LLC",FAM-NY-2719389,NY-2719389,screened_singleton_not_proof_of_independence,False,no_known_boundary_conflict
NY-2757329,Heck v. Regula,FAM-NY-2757329,NY-2757329,screened_singleton_not_proof_of_independence,True,no_known_boundary_conflict


## Outputs and remaining decisions

- `data/family_v1/family_manifest.json`: all 495 observed families across 506 sources.
- `case_family_manifest.jsonl`: one row per source with family, period flags, and exposure.
- `chunk_family_manifest.jsonl`: all 1,954 existing source passages mapped to families.
- `annotation_packet_membership.jsonl`: 506 per-case memberships in original packets.
- `pair_review_log.jsonl`, `opinion_notice_review.jsonl`, `uncertain_matches.jsonl`:
  reviewed evidence, rejected candidates, and unresolved publication/companion issues.
- `screening_results.json`, `summary.json`, `provenance.json`: settings and checksums.
- `data/metadata_v2_family_review/metadata.jsonl`: updated 114-case metadata;
  supersedes `metadata_v1` for family/calibration information, not target adjudication.

Human review should confirm grouping decisions, resolve Sodhi's missing companion
and canonical publication choices, and confirm any additional calibration exposure.
The future split validator checks family/calibration/companion constraints; it does
not implement the complete chronological split or replace target-review checks.
Any new source, relationship, or decision requires a new version and regeneration.
